In [35]:
import sympy as sp

# Define symbols
alpha = sp.Symbol('alpha')
k1, k2, k3 = sp.symbols('k1 k2 k3')
mu_X, mu_Y = sp.symbols('mu_X mu_Y')
sigma_X2, sigma_Y2, sigma_XY = sp.symbols('sigma_X2 sigma_Y2 sigma_XY')
S_XXX, S_YYY, S_XXY, S_XYY = sp.symbols('S_XXX S_YYY S_XXY S_XYY')

# Define mean, variance, and skewness as functions of alpha
mu_alpha = alpha * mu_X + (1 - alpha) * mu_Y

sigma2_alpha = alpha**2 * sigma_X2 + (1 - alpha)**2 * sigma_Y2 + 2 * alpha * (1 - alpha) * sigma_XY

S_alpha = (alpha**3 * S_XXX + (1 - alpha)**3 * S_YYY +
           3 * alpha**2 * (1 - alpha) * S_XXY +
           3 * alpha * (1 - alpha)**2 * S_XYY)

# Define the function F(alpha)
F_alpha = k1 * mu_alpha - k2 * sigma2_alpha + k3 * S_alpha

# Display the equation

F_alpha

k1*(alpha*mu_X + mu_Y*(1 - alpha)) - k2*(alpha**2*sigma_X2 + 2*alpha*sigma_XY*(1 - alpha) + sigma_Y2*(1 - alpha)**2) + k3*(S_XXX*alpha**3 + 3*S_XXY*alpha**2*(1 - alpha) + 3*S_XYY*alpha*(1 - alpha)**2 + S_YYY*(1 - alpha)**3)

In [36]:
# Rewrite as polynomial in alpha
expansion = sp.expand(F_alpha)
collection = sp.collect(expansion, alpha)
sp.diff(collection, alpha).simplify()


3*S_XYY*k3 - 3*S_YYY*k3 + 3*alpha**2*k3*(S_XXX - 3*S_XXY + 3*S_XYY - S_YYY) + 2*alpha*(3*S_XXY*k3 - 6*S_XYY*k3 + 3*S_YYY*k3 - k2*sigma_X2 + 2*k2*sigma_XY - k2*sigma_Y2) + k1*mu_X - k1*mu_Y - 2*k2*sigma_XY + 2*k2*sigma_Y2

In [37]:
# Take the derivative of F(alpha) with respect to alpha and simplify
derivative = sp.diff(F_alpha, alpha)
simplified_derivative = derivative.simplify()
der = sp.expand(simplified_derivative)
sp.collect(der, alpha).simplify()

3*S_XYY*k3 - 3*S_YYY*k3 + 3*alpha**2*k3*(S_XXX - 3*S_XXY + 3*S_XYY - S_YYY) + 2*alpha*(3*S_XXY*k3 - 6*S_XYY*k3 + 3*S_YYY*k3 - k2*sigma_X2 + 2*k2*sigma_XY - k2*sigma_Y2) + k1*mu_X - k1*mu_Y - 2*k2*sigma_XY + 2*k2*sigma_Y2

In [44]:
# Rewrite as polynomial in alpha
expansion = sp.expand(F_alpha)
collection = sp.collect(expansion, alpha)
derivative = sp.diff(collection, alpha).simplify()
derivative

# factor
# sp.factor(derivative)
sp.solve(derivative, alpha)[0].simplify()


(-3*S_XXY*k3 + 6*S_XYY*k3 - 3*S_YYY*k3 + k2*sigma_X2 - 2*k2*sigma_XY + k2*sigma_Y2 - sqrt(-9*S_XXX*S_XYY*k3**2 + 9*S_XXX*S_YYY*k3**2 - 3*S_XXX*k1*k3*mu_X + 3*S_XXX*k1*k3*mu_Y + 6*S_XXX*k2*k3*sigma_XY - 6*S_XXX*k2*k3*sigma_Y2 + 9*S_XXY**2*k3**2 - 9*S_XXY*S_XYY*k3**2 - 9*S_XXY*S_YYY*k3**2 + 9*S_XXY*k1*k3*mu_X - 9*S_XXY*k1*k3*mu_Y - 6*S_XXY*k2*k3*sigma_X2 - 6*S_XXY*k2*k3*sigma_XY + 12*S_XXY*k2*k3*sigma_Y2 + 9*S_XYY**2*k3**2 - 9*S_XYY*k1*k3*mu_X + 9*S_XYY*k1*k3*mu_Y + 12*S_XYY*k2*k3*sigma_X2 - 6*S_XYY*k2*k3*sigma_XY - 6*S_XYY*k2*k3*sigma_Y2 + 3*S_YYY*k1*k3*mu_X - 3*S_YYY*k1*k3*mu_Y - 6*S_YYY*k2*k3*sigma_X2 + 6*S_YYY*k2*k3*sigma_XY + k2**2*sigma_X2**2 - 4*k2**2*sigma_X2*sigma_XY + 2*k2**2*sigma_X2*sigma_Y2 + 4*k2**2*sigma_XY**2 - 4*k2**2*sigma_XY*sigma_Y2 + k2**2*sigma_Y2**2))/(3*k3*(S_XXX - 3*S_XXY + 3*S_XYY - S_YYY))

In [41]:
import pandas as pd
import numpy as np

data = pd.read_csv('/Users/ivankhalin/Documents/code/speed_test/data/info.csv', index_col=0, usecols=[0, 1, 2, 3], skiprows=1, parse_dates=False, names=['date', 'DAX', 'S&P', 'rate'])
data.index = pd.DatetimeIndex(data.index, freq='W-MON')
data = data[data.index.year < 2024]
data['rate'] = data['rate'] / 100 / 52
returns = data.pct_change().dropna()

# Compute weekly returns and residuals
# data = returns[['DAX', 'S&P']].reset_index()
# data.set_index('Date', inplace=True)

# Calculate monthly returns
returns = data.pct_change().dropna()

# Mean returns
mu_X = returns['DAX'].mean()
mu_Y = returns['S&P'].mean()

# Variances
sigma_X2 = returns['DAX'].var()
sigma_Y2 = returns['S&P'].var()

# Covariance
sigma_XY = returns['DAX'].cov(returns['S&P'])

# Skewness
S_XXX = returns['DAX'].skew()
S_YYY = returns['S&P'].skew()

# Co-skewness
S_XXY = ((returns['DAX'] - mu_X)**2 * (returns['S&P'] - mu_Y)).mean() / (sigma_X2**1.5)
S_XYY = ((returns['DAX'] - mu_X) * (returns['S&P'] - mu_Y)**2).mean() / (sigma_Y2**1.5)
parameters = {
    "mu_X": mu_X, "mu_Y": mu_Y,
    "sigma_X2": sigma_X2, "sigma_Y2": sigma_Y2, "sigma_XY": sigma_XY,
    "S_XXX": S_XXX, "S_YYY": S_YYY, "S_XXY": S_XXY, "S_XYY": S_XYY
}

print(parameters)

{'mu_X': 0.0015407944467243806, 'mu_Y': 0.0017717220518074577, 'sigma_X2': 0.0012112456901435375, 'sigma_Y2': 0.0006660012795442428, 'sigma_XY': 0.0007094414650889019, 'S_XXX': -0.34581508242489073, 'S_YYY': -0.09890625145095208, 'S_XXY': -0.1947987839214922, 'S_XYY': -0.23844777786569726}
